# **Plotting Script: ENSO Transition Probability Experiment Split by Eruption Seasonality**

Output:

1) "/home/563/ft3359/FT-Honours/Honours_Paper/Figures/TransProb_season.png"

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D


# FONT
mpl.rcParams["font.family"]     = "sans-serif"
mpl.rcParams["font.sans-serif"] = ["Arial", "Helvetica", "DejaVu Sans"]


# INPUT / OUTPUT
TABLE_DIR = "/home/563/ft3359/FT-Honours/Honours_Paper/Transition_Prob/Seasonal_Analysis"
MODEL_CSV = os.path.join(TABLE_DIR, "TransProb_Models_Seasonal.csv")
RECON_CSV = os.path.join(TABLE_DIR, "TransProb_Recons_Seasonal.csv")
OUT_PNG   = "/home/563/ft3359/FT-Honours/Honours_Paper/Figures/TransProb_seasonal.png"


# SETTINGS
PHASES = ["El Niño", "Neutral", "La Niña"]
SEASON_ORDER = ["DJF", "MAM", "JJA", "SON"]

# Colour Set
PHASE_COLORS = {
    "El Niño": "#D55E00",
    "Neutral": "#A8B4C0",
    "La Niña": "#0072B2",
}

ANNOTATE_MIN_FRAC = 0.05

# Donut geometry
OUTER_RADIUS  = 1.63
INNER_RADIUS  = 1.09
RING_WIDTH    = 0.53
ERUPT_LABEL_R = 1.37
BASE_LABEL_R  = 0.83

# Font sizes
PCT_FONTSIZE           = 16
TITLE_FONTSIZE          = 19
ROW_LABEL_FONTSIZE      = 17
LEGEND_FONTSIZE         = 15
LEGEND_TITLE_FONTSIZE   = 16
SEASON_LABEL_FONTSIZE   = 17
GROUP_LABEL_FONTSIZE    = 19
PANEL_LABEL_FONTSIZE    = 20

# HELPERS

def sanitize_prob_vector(p):
    p = np.asarray(p, dtype=float).copy()
    p[~np.isfinite(p)] = 0.0
    p[p < 0]           = 0.0
    s = p.sum()
    if s > 0:
        p /= s
    else:
        p[:] = 0.0
    return p


def read_wlp_csv_by_season(csv_path, season):
    df = pd.read_csv(csv_path)
    needed = {
        "season", "start_phase",
        "baseline_EN", "baseline_N", "baseline_LN",
        "eruption_EN", "eruption_N", "eruption_LN",
    }
    missing = sorted(needed - set(df.columns))
    if missing:
        raise KeyError(f"CSV missing columns {missing}: {csv_path}")

    df_season = df[df["season"] == season]

    probs_base  = {}
    probs_erupt = {}

    for sp in PHASES:
        row = df_season[df_season["start_phase"] == sp]
        if row.empty:
            probs_base[sp]  = np.zeros(3, dtype=float)
            probs_erupt[sp] = np.zeros(3, dtype=float)
            continue
        r = row.iloc[0]
        probs_base[sp]  = sanitize_prob_vector(
            [r["baseline_EN"], r["baseline_N"], r["baseline_LN"]]
        )
        probs_erupt[sp] = sanitize_prob_vector(
            [r["eruption_EN"], r["eruption_N"], r["eruption_LN"]]
        )

    return probs_base, probs_erupt


def donut_on_ax(ax, pB, pE, title):
    colors = [PHASE_COLORS["El Niño"], PHASE_COLORS["Neutral"], PHASE_COLORS["La Niña"]]

    if pB.sum() <= 0 and pE.sum() <= 0:
        ax.text(0, 0, "No transitions", ha="center", va="center", fontsize=PCT_FONTSIZE)
        ax.set_title(title, fontsize=TITLE_FONTSIZE)
        ax.set_aspect("equal")
        ax.axis("off")
        return

    # Outer ring = eruption
    wedgesE = []
    if pE.sum() > 0:
        wedgesE, _ = ax.pie(
            pE,
            radius=OUTER_RADIUS,
            colors=colors,
            startangle=90,
            wedgeprops=dict(width=RING_WIDTH, edgecolor="white"),
        )

    # Inner ring = baseline
    wedgesB = []
    if pB.sum() > 0:
        wedgesB, _ = ax.pie(
            pB,
            radius=INNER_RADIUS,
            colors=colors,
            startangle=90,
            wedgeprops=dict(width=RING_WIDTH, edgecolor="white"),
        )

    # Percentage labels
    for i in range(3):
        if i < len(wedgesE) and pE[i] >= ANNOTATE_MIN_FRAC:
            ang = (wedgesE[i].theta1 + wedgesE[i].theta2) / 2
            ax.text(
                ERUPT_LABEL_R * np.cos(np.deg2rad(ang)),
                ERUPT_LABEL_R * np.sin(np.deg2rad(ang)),
                f"{100*pE[i]:.0f} %",
                ha="center", va="center",
                fontsize=PCT_FONTSIZE, color="white", weight="bold",
            )

        if i < len(wedgesB) and pB[i] >= ANNOTATE_MIN_FRAC:
            ang = (wedgesB[i].theta1 + wedgesB[i].theta2) / 2
            ax.text(
                BASE_LABEL_R * np.cos(np.deg2rad(ang)),
                BASE_LABEL_R * np.sin(np.deg2rad(ang)),
                f"{100*pB[i]:.0f} %",
                ha="center", va="center",
                fontsize=PCT_FONTSIZE, color="white", weight="bold",
            )

    lim = OUTER_RADIUS + 0.09
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_title(title, fontsize=TITLE_FONTSIZE, pad=6)
    ax.set_aspect("equal")
    ax.axis("off")

# LOAD DATA

for p in [MODEL_CSV, RECON_CSV]:
    if not os.path.isfile(p):
        raise FileNotFoundError(f"Missing input CSV: {p}")

data_by_season = {}
for season in SEASON_ORDER:
    model_base, model_erupt = read_wlp_csv_by_season(MODEL_CSV, season)
    recon_base, recon_erupt = read_wlp_csv_by_season(RECON_CSV, season)
    data_by_season[season] = dict(
        model_base=model_base, model_erupt=model_erupt,
        recon_base=recon_base, recon_erupt=recon_erupt,
    )


# PLOT — 4 rows (seasons) x 6 columns (Models' 3 phases | Reconstructions' 3 phases)

n_rows = len(SEASON_ORDER)
n_cols = len(PHASES) * 2

fig, axes = plt.subplots(n_rows, n_cols, figsize=(28, 5.3 * n_rows), dpi=300)

for si, season in enumerate(SEASON_ORDER):
    d = data_by_season[season]
    show_title = (si == 0)

    for j, sp in enumerate(PHASES):
        title = f"Start: {sp}" if show_title else ""
        donut_on_ax(
            axes[si, j],
            d["model_base"].get(sp,  np.zeros(3)),
            d["model_erupt"].get(sp, np.zeros(3)),
            title=title,
        )

    for j, sp in enumerate(PHASES):
        title = f"Start: {sp}" if show_title else ""
        donut_on_ax(
            axes[si, j + 3],
            d["recon_base"].get(sp,  np.zeros(3)),
            d["recon_erupt"].get(sp, np.zeros(3)),
            title=title,
        )

    # Season row label, to the left of the leftmost panel
    axes[si, 0].annotate(
        season, xy=(-0.38, 0.5), xycoords="axes fraction",
        fontsize=SEASON_LABEL_FONTSIZE, fontweight="bold",
        ha="center", va="center", rotation=90,
    )

fig.subplots_adjust(left=0.06, right=0.87, top=0.90, bottom=0.02, wspace=0.24, hspace=0.28)

# Group headers spanning each half (Models / Reconstructions)
fig.text(0.30, 0.95, "Models", ha="center", fontsize=GROUP_LABEL_FONTSIZE, fontweight="bold")
fig.text(0.68, 0.95, "Reconstructions", ha="center", fontsize=GROUP_LABEL_FONTSIZE, fontweight="bold")

fig.text(0.06, 0.985, "(a)", ha="left", va="top", fontsize=PANEL_LABEL_FONTSIZE, fontweight="bold")
fig.text(0.465, 0.985, "(b)", ha="left", va="top", fontsize=PANEL_LABEL_FONTSIZE, fontweight="bold")


# Legend
color_handles = [Rectangle((0, 0), 1, 1, color=PHASE_COLORS[ph])
                 for ph in ["El Niño", "Neutral", "La Niña"]]
color_labels  = ["El Niño", "Neutral", "La Niña"]

blank         = Line2D([], [], linestyle="none", marker="none")
ring_handles  = [blank, blank]
ring_labels   = ["Outer ring = volcanic", "Inner ring = baseline"]

all_handles = color_handles + ring_handles
all_labels  = color_labels  + ring_labels

leg = fig.legend(
    all_handles,
    all_labels,
    title="Next ENSO phase",
    loc="center right",
    bbox_to_anchor=(1.05, 0.5),
    fontsize=LEGEND_FONTSIZE,
    title_fontsize=LEGEND_TITLE_FONTSIZE,
    labelspacing=1.2,
    handlelength=1.8,
    handleheight=1.8,
    handletextpad=0.8,
    borderpad=1.2,
    frameon=True,
)

for handle, text in zip(leg.legend_handles, leg.get_texts()):
    if text.get_text() in ring_labels:
        handle.set_visible(False)

os.makedirs(os.path.dirname(OUT_PNG), exist_ok=True)
fig.savefig(OUT_PNG, dpi=300, bbox_inches="tight")
print("Saved:", OUT_PNG)

plt.show()

Saved: /home/563/ft3359/FT-Honours/Honours_Paper/Figures/TransProb_seasonal.png
